# Environment verification — M0-SETUP-01

Closes the acceptance criteria of `M0-SETUP-01` that cannot be checked from a
laptop: **`pip install -r requirements.txt` succeeds in a fresh cell, with no
manual fixes** (`spec/smoke.md §1`).

Run every cell top to bottom on a fresh runtime, then paste the output of cells
2, 4 and 5 into a new `AGENT_LOG.md` entry. Executed copies are kept as dated
evidence in `notebooks/runs/`.

No project logic lives here — this notebook only reports and calls
(`spec/architecture.md §2`). It is a verification harness, not part of the
pipeline. It also retypes no constants: requirements are read from the file and
decoder parameters from `protocol.py`, so it cannot drift from the spec.

## Getting the repo onto the runtime

**Mounting Google Drive is the primary path**, and cell 3 does it for you. Put
the repo in Drive (e.g. `MyDrive/qecscreen`) and it will be found; set the
`QECSCREEN_REPO` environment variable to override the search.

> **`files.upload()` does not work under Colab-in-VS-Code.** The file picker is
> a browser-side widget and is refused when Colab is attached as a VS Code
> kernel, which is how this project normally runs it. It survives only as an
> optional browser-Colab-only cell at the bottom. Do not rely on it.

## What this is actually testing

A local venv starts empty, so a clean install there proves very little. Colab
and Kaggle arrive with their own numpy, pandas, scipy and scikit-learn. The real
questions are whether our bounds force a **downgrade** of that preinstalled
stack, whether a **C compiler** gets invoked, and whether **any manual fix** is
needed. Cell 4 checks all three.

In [ ]:
# 2 — what interpreter are we on?
import sys, platform

print("Python  :", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    import google.colab  # noqa: F401
    print("Runtime : Google Colab")
except ImportError:
    print("Runtime : not Colab (local kernel, Kaggle, or other host)")

# Both 3.11 and 3.13 are supported target environments: Kaggle runs 3.11,
# Colab runs 3.13 (spec/architecture.md section 1). Record which one this was.
print()
print("NOTE: record this Python version with the run.")

In [ ]:
# 3 — locate requirements.txt. Drive mount is the primary path.
import os, pathlib

REPO_ENV = "QECSCREEN_REPO"  # set this to override the search


def _first_with_requirements(roots):
    for root in roots:
        req = root / "requirements.txt"
        if req.is_file():
            return req.resolve()
    return None


def locate_requirements():
    override = os.environ.get(REPO_ENV)
    if override:
        found = _first_with_requirements([pathlib.Path(override).expanduser()])
        if found:
            return found

    # Already reachable? Local kernel, Kaggle input, or a repo copied onto the
    # runtime. Walk up from the working directory.
    here = pathlib.Path.cwd()
    found = _first_with_requirements([here, *here.parents])
    if found:
        return found

    # Primary Colab path: mount Drive and search the usual layouts.
    try:
        from google.colab import drive
    except ImportError:
        return None
    drive.mount("/content/drive")

    bases = [
        pathlib.Path("/content/drive/MyDrive"),
        pathlib.Path("/content/drive/Shareddrives"),
    ]
    candidates = []
    for base in bases:
        if base.exists():
            candidates += [base, base / "qecscreen", base / "qecscreen" / "qecscreen"]
    found = _first_with_requirements(candidates)
    if found:
        return found

    # Last resort: search Drive for a qecscreen checkout.
    for base in bases:
        if base.exists():
            for candidate in base.rglob("requirements.txt"):
                if "qecscreen" in str(candidate).lower():
                    return candidate.resolve()
    return None


REQ = locate_requirements()
if REQ is None:
    raise RuntimeError(
        "requirements.txt not found. Put the repo in Drive under MyDrive/qecscreen, "
        f"or set {REPO_ENV} to its path. If you are in a browser Colab tab you may "
        "use the optional upload cell at the bottom instead."
    )

REPO = REQ.parent
print("Using repo root  :", REPO)
print("Using requirements:", REQ)
print()
print(REQ.read_text(encoding="utf-8"))

In [ ]:
# 4 — the acceptance criterion, and the three things that can fail it
import re, subprocess, sys

proc = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(REQ)],
    capture_output=True,
    text=True,
)
log = proc.stdout + proc.stderr
print("pip exit code:", proc.returncode)
print()

for line in log.splitlines():
    if any(t in line for t in ("Successfully installed", "Attempting uninstall",
                               "Successfully uninstalled", "Building wheel",
                               "ERROR", "incompatible")):
        print(line)
print()

# (a) did the preinstalled scientific stack get downgraded?
AMBIENT = ("numpy", "pandas", "scipy", "scikit-learn", "scikit_learn")
clobbered = [
    line.strip()
    for line in log.splitlines()
    if "Attempting uninstall" in line and any(p in line.lower() for p in AMBIENT)
]

# (b) did a C toolchain run? Same test the CI guard uses. A pure-Python sdist
#     build (sinter does one) is fine; a C build is not, because Kaggle and
#     Colab must install this in one cell without a compiler.
COMPILER = re.compile(
    r"x86_64-linux-gnu-gcc|(^|\s)(gcc|clang|cc1)\s+-|Microsoft Visual C\+\+"
    r"|error: command .* failed with exit",
    re.I | re.M,
)
compiled = COMPILER.search(log)

print("pip exit code 0          :", proc.returncode == 0)
print("no ambient downgrade     :", not clobbered, clobbered or "")
print("no compiler invocation   :", not compiled,
      compiled.group(0).strip() if compiled else "")
print()
print("THIRD CRITERION IS YOURS: did you have to add any cell of your own to")
print("make this work? If so, spec/smoke.md section 1 ('no manual fixes') is NOT")
print("met. Record the cell verbatim and add the constraint to requirements.txt")
print("instead - do not soften the criterion.")

assert proc.returncode == 0, "pip install FAILED - paste the full log into AGENT_LOG.md"
assert not clobbered, f"downgraded the preinstalled stack: {clobbered}"
assert not compiled, f"a C toolchain was invoked: {compiled.group(0)!r}"

In [ ]:
# 5 — what actually resolved
import importlib
from importlib.metadata import version, PackageNotFoundError

importlib.invalidate_caches()

PINNED = [
    "numpy", "scipy", "pandas", "pyarrow", "networkx",
    "stim", "sinter", "ldpc",
    "scikit-learn", "lightgbm", "pytest",
    "matplotlib", "pymatching",
]
for name in PINNED:
    try:
        print(name.ljust(14), version(name))
    except PackageNotFoundError:
        print(name.ljust(14), "MISSING")

print()
print("stim/sinter/ldpc versions enter protocol_hash (INV-6) - record them.")

In [ ]:
# 6 — every import must be warning-free, because pytest.ini sets filterwarnings = error
import subprocess, sys

check = subprocess.run(
    [
        sys.executable, "-W", "error", "-c",
        "import stim, sinter, ldpc, networkx, pandas, numpy, scipy, sklearn, lightgbm, pyarrow; "
        "from ldpc import BpOsdDecoder; "
        "from ldpc.sinter_decoders import SinterBpOsdDecoder; "
        "print('imports clean under -W error')",
    ],
    capture_output=True,
    text=True,
)
print(check.stdout or check.stderr)
assert check.returncode == 0, "an import emitted a warning - pytest would fail on this runtime"

In [ ]:
# 7 — do the pinned CONTRACT decoder parameters work on this runtime?
# Parameters are read from protocol.py, never retyped here (INV-6 / no drift).
import sys

src = REPO / "src"
if not src.is_dir():
    print("src/ is not next to requirements.txt - skipping.")
else:
    sys.path.insert(0, str(src))
    import numpy as np
    from ldpc import BpOsdDecoder
    from qecscreen.protocol import DECODER_PARAMS

    kwargs = {k: v for k, v in DECODER_PARAMS.items() if k not in ("library", "decoder")}
    H = np.array([[1, 1, 0], [0, 1, 1]], dtype=np.uint8)
    dec = BpOsdDecoder(H, error_rate=0.01, **kwargs)
    print("BpOsdDecoder accepted:", kwargs)
    print("osd_method as the library stores it:", dec.osd_method)

In [ ]:
# 8 — the full suite, if the repo source is reachable
import subprocess, sys

if not (REPO / "tests").is_dir():
    print("tests/ not reachable - skipping. Cells 4 and 5 are the acceptance criterion.")
else:
    run = subprocess.run(
        [sys.executable, "-m", "pytest"],
        cwd=str(REPO), capture_output=True, text=True,
    )
    print(run.stdout[-3000:])
    print(run.stderr[-2000:])
    print("pytest exit code:", run.returncode)

## Recording the result

Save the executed copy to `notebooks/runs/<YYYY-MM-DD>-verify-env-<host>.ipynb`
and paste the output of cells 2, 4 and 5 into a new `AGENT_LOG.md` entry. The
frozen run from 2026-08-30 is the reason the `sinter` wheel error was caught, so
these files earn their keep.

The criterion is met when **all four** hold:

1. cell 4 prints `pip exit code: 0`;
2. **no ambient downgrade** — nothing uninstalled numpy, pandas, scipy or
   scikit-learn. A downgrade means our bounds are fighting the host;
3. **no compiler invocation** — a pure-Python sdist build is fine and expected
   (`sinter` 1.16.0 is sdist-only), a C build is not, because it needs a
   toolchain the one-cell install cannot assume;
4. **no manual fixes** — you added no cell of your own. Cell 4 cannot check this
   one; only you know.

If 4 fails, the fix is to add the missing constraint to `requirements.txt` so a
single cell suffices. **Do not soften the criterion** — `spec/evals.md §7` says
a PARTIAL is never rounded up.

Still outstanding: this has **never been run on Kaggle**, which is the machine
that does the bulk generation. `spec/smoke.md §1` requires it.

---

### Optional: upload instead of mounting Drive — browser Colab only

Do **not** use this under Colab-in-VS-Code; the picker is a browser-side widget
and is refused there. Run it only in a browser Colab tab, before cell 4, if you
would rather not mount Drive.

```python
from google.colab import files
import pathlib
uploaded = files.upload()          # choose requirements.txt
REQ = pathlib.Path(next(iter(uploaded))).resolve()
REPO = REQ.parent                  # cells 7 and 8 will then skip themselves
```